In [14]:
import pandas as pd
import numpy as np
from shapely import wkt
import geopandas as gpd
import requests, time
from rapidfuzz import fuzz

In [ ]:
ind_landmarks_df = pd.read_csv("source_csvs/Individual_Landmark_Sites_20260725.csv")
# landmarks_districts_df = pd.read_csv("source_csvs/Individual_Landmark_and_Historic_District_Building_Database_20260801.csv")
# scenic_landmarks_df = pd.read_csv("source_csvs/Scenic_Landmarks_20260801.csv")
# historic_districts_df = pd.read_csv("source_csvs/Historic_Districts_20260801.csv")
neighborhoods_df = pd.read_csv("source_csvs/2020_Neighborhood_Tabulation_Areas_(NTAs)_20260802.csv")

In [ ]:
# landmarks_districts_df = landmarks_districts_df.drop(['the_geom', 'Borough', 'Block', 
#                                                       'Lot', 'DESIG_ADDRESS', 'LM_NAME2', 
#                                                       'shape_leng', 'shape_area'], 
#                                                       axis=1)

# landmarks_districts_df =landmarks_districts_df[landmarks_districts_df['LM_NAME'] != '0']

In [ ]:
# print("Landmark and Historic Building Sites:", len(landmarks_districts_df['LM_NAME'].unique()))
# print("Individual Landmarks:", len(ind_landmarks_df['LM_NAME'].unique()))
# print("Scenic Landmarks:", len(scenic_landmarks_df['LM_NAME'].unique()))

Landmark and Historic Building Sites: 1451
Individual Landmarks: 1504
Scenic Landmarks: 8


In [6]:
combined_building_landmarks = ind_landmarks_df.merge(landmarks_districts_df, on='LM_NAME', how='left')

In [7]:
neighborhoods_df['the_geom'] = neighborhoods_df['the_geom'].apply(wkt.loads)

combined_building_landmarks['the_geom'] = combined_building_landmarks['the_geom'].apply(wkt.loads)

In [8]:
# neighborhoods_gdf = gpd.GeoDataFrame(neighborhoods_df, geometry='the_geom', crs='EPSG:2263')
landmarks_gdf = gpd.GeoDataFrame(combined_building_landmarks, geometry='the_geom', crs='EPSG:2263')

In [9]:
# This new dataset is lat/lon -> EPSG:4326
neighborhoods_gdf = gpd.GeoDataFrame(neighborhoods_df, geometry='the_geom', crs='EPSG:4326')

# Reproject neighborhoods into the landmarks' CRS (EPSG:2263) so they match
neighborhoods_gdf = neighborhoods_gdf.to_crs('EPSG:2263')

In [10]:
# landmarks_gdf = landmarks_gdf.to_crs(neighborhoods_gdf.crs)

# 4. For each landmark, find the neighborhood it falls inside
def find_neighborhood(landmark_geom):
    matches = neighborhoods_gdf[neighborhoods_gdf.contains(landmark_geom)]
    if not matches.empty:
        return matches.iloc[0]['NTAName']  # adjust to your actual name column
    return None

landmarks_gdf['neighborhood'] = landmarks_gdf['the_geom'].apply(find_neighborhood)

In [ ]:
HEADERS = {"User-Agent": "landmark-fame-project (sean.cary62@gmail.com)"}  

def geosearch_wikipedia(lat, lon, radius_m=150, limit=5):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "list": "geosearch",
        "gscoord": f"{lat}|{lon}", "gsradius": radius_m,
        "gslimit": limit, "format": "json"
    }, headers=HEADERS)
    return r.json().get("query", {}).get("geosearch", [])

In [17]:
def best_match(landmark_name, candidates):
    if not candidates:
        return None, 0
    scored = [(c, fuzz.token_sort_ratio(landmark_name, c["title"])) for c in candidates]
    scored.sort(key=lambda x: -x[1])
    return scored[0]  # (candidate dict, similarity score 0-100)

In [65]:
def resolve_title(title):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "titles": title, "redirects": 1, "format": "json"
    }, headers=HEADERS)
    pages = r.json().get("query", {}).get("pages", {})
    return next(iter(pages.values())).get("title", title)

def get_pageviews(title, start="20240101", end="20241231"):
    print(title)
    title_enc = resolve_title(title).replace(" ", "_")
    r = requests.get(
        f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia.org/all-access/user/{title_enc}/monthly/{start}/{end}",
        headers=HEADERS
    )
    if r.status_code != 200:
        return None
    return sum(item["views"] for item in r.json().get("items", []))

def resolve_title_and_url(title):
    r = requests.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query", "titles": title, "redirects": 1,
        "prop": "info", "inprop": "url", "format": "json"
    }, headers=HEADERS)
    pages = r.json().get("query", {}).get("pages", {})
    page = next(iter(pages.values()))
    return {
        "title": page.get("title", title),
        "url": page.get("fullurl")  # None if the page doesn't exist
    }

In [70]:
landmarks_gdf.columns

Index(['the_geom', 'OBJECTID', 'Borough', 'Block', 'Lot', 'DESIG_ADDRESS',
       'BBL_x', 'LM_NAME', 'LP_NUMBER', 'SITE_DESC', 'SITE_STATUS', 'LM_NAME2',
       'DESIG_DATE', 'LM_TYPE', 'REPORT_URL', 'CD', 'Council', 'Latitude',
       'Longitude', 'BCT2020', 'NTA2020', 'Shape_Leng', 'Shape_Area', 'BIN',
       'BBL_y', 'CIRCA', 'DATE_LOW', 'DATE_HIGH', 'DATE_COMBO', 'ALT_DATE1',
       'ALT_DATE2', 'ARCHITECT', 'HIST_OWNER', 'ALT_ARCHITECT',
       'ALT_ARCHITECT2', 'ALTERED', 'STYLE1', 'STYLE2', 'STYLE3', 'MATERIAL1',
       'MATERIAL2', 'MATERIAL3', 'MATERIAL4', 'MATERIAL5', 'ORIG_USE',
       'ORIG_USE2', 'BUILD_TYPE', 'BUILD_TYPE2', 'PROP_NAME', 'LM_NOTES',
       'HIST_DISTRICT', 'neighborhood'],
      dtype='object')

In [79]:
def match_and_score(row, radius_m=150, name_threshold=70):
    candidates = geosearch_wikipedia(row["Latitude"], row["Longitude"], radius_m=radius_m)
    match, score = best_match(row["LM_NAME"], candidates)

    if match is None or score < name_threshold:
        r = requests.get("https://en.wikipedia.org/w/api.php", params={
            "action": "query", "list": "search", "srsearch": row["LM_NAME"],
            "srlimit": 3, "format": "json"
        }, headers=HEADERS)
        text_candidates = [{"title": s["title"]} for s in r.json().get("query", {}).get("search", [])]
        match, score = best_match(row["LM_NAME"], text_candidates)
        confidence = "low" if match else "none"
    else:
        confidence = "high" if score > 85 else "medium"

    if match is None:
        return {"wiki_title": None, "wiki_url": None, "confidence": "none", "pageviews": None, "score":score}

    resolved = resolve_title_and_url(match["title"])
    views = get_pageviews(resolved["title"])
    return {
        "wiki_title": resolved["title"],
        "wiki_url": resolved["url"],
        "confidence": confidence,
        "pageviews": views,
        "score": score
    }

In [80]:
test= match_and_score(landmarks_gdf.iloc[542], radius_m=150, name_threshold=50)

Prospect Park (Brooklyn)


In [81]:
print(test)

{'wiki_title': 'Prospect Park (Brooklyn)', 'wiki_url': 'https://en.wikipedia.org/wiki/Prospect_Park_(Brooklyn)', 'confidence': 'low', 'pageviews': 68991, 'score': 27.500000000000004}


In [82]:
df_sample = landmarks_gdf.sample(n=5, random_state=42)

In [92]:
for idx, row in landmarks_gdf.iterrows():
    try:
        result = match_and_score(row, radius_m=100, name_threshold=50)
        landmarks_gdf.at[idx, 'wiki_title'] = result['wiki_title']
        landmarks_gdf.at[idx, 'wiki_url'] = result['wiki_url']
        landmarks_gdf.at[idx, 'confidence'] = result['confidence']
        landmarks_gdf.at[idx, 'pageviews'] = result['pageviews']
        landmarks_gdf.at[idx, 'score'] = result['score']
    except Exception as e:
        print(f"Error processing row {idx}: {e}")
    time.sleep(0.5) 

St. Mary's Church
Tarlac National High School
Lithuanian Alliance of America
Abraham E. Lefcourt
Daniel E. Barbey
List of New York City Designated Landmarks in Manhattan from 14th to 59th Streets
601 West 29th Street
Eiffel Tower
601 West 29th Street
Whitney Museum of American Art
Daniel Stern (actor)
Frederick Douglass Memorial Park
Brooklyn Friends School
Crown Building (Manhattan)
List of New York Public Library branches
Modulightor Building
Ulrich Franzen
African Burial Ground National Monument
Minton's Playhouse
Cartier Building
List of New York City Designated Landmarks in Queens
Los Angeles Fire Department
Bronx Opera House
New York City Fire Department
Simmons Colored School
Julius (restaurant)
Samuel Gompers High School
Lesbian Herstory Archives
Benjamin Ralph Kimlau
List of New York City Designated Landmarks in Staten Island
New York Public Library Main Branch
Holyrood Episcopal Church
Flatiron Building
List of New York City Designated Landmarks in Brooklyn
Guardian Angels
Vo

In [93]:
landmarks_gdf.to_csv("output_csvs/landmarks_with_neighborhoods.csv", index=False)